# Task 01 – Python Programming, Data Analysis & Visualization
**Devixo Solutions – AI/ML Internship Program**

**Dataset:** Diabetes 130 US Hospitals for Years 1999–2008 (Kaggle: `brandao/diabetes`)
**Author:** *<your name>*

---
## PART 1 – Data Loading & Data Understanding

### 1.1 Import Required Libraries
- **pandas** – load and work with tables (DataFrames)
- **numpy** – numerical operations
- **matplotlib / seaborn** – charts (used later in Part 4)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Show all columns when printing tables (this dataset has 50 columns)
pd.set_option("display.max_columns", None)

# Print library versions so the results can be reproduced
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)

pandas : 2.2.3
numpy  : 2.1.3


### 1.2 Load the Dataset
**Loading decisions:**
- `keep_default_na=False` – by default pandas turns the text `"None"` into NaN. In this dataset `"None"` in `max_glu_serum` and `A1Cresult` means *the test was not performed*, which is real information. We keep the raw text and handle the hidden missing values (`"?"`) deliberately in Part 2.
- `low_memory=False` – makes pandas read the whole column before deciding its type, which avoids mixed-type warnings.
- **Integrity check** – the original file has 101,766 rows and 50 columns. If the loaded file is different (for example an incomplete upload), the notebook stops here instead of producing wrong results.

In [3]:
from pathlib import Path

# The notebook looks for the file in these places (local repo layout first, then Google Colab upload)
candidates = [
    Path("../dataset/diabetic_data.csv"),
    Path("dataset/diabetic_data.csv"),
    Path("diabetic_data.csv"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("diabetic_data.csv not found. Current folder: " + str(Path.cwd()))

df = pd.read_csv(DATA_PATH, keep_default_na=False, low_memory=False)
print("Loaded from:", DATA_PATH)

# ---- Integrity check against the published dataset size ----
EXPECTED_SHAPE = (101766, 50)
EXPECTED_NUMERIC = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
                    "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]

assert df.shape == EXPECTED_SHAPE, (
    f"Unexpected shape {df.shape}; expected {EXPECTED_SHAPE}. The file may be incomplete or modified - re-download it.")
bad = [c for c in EXPECTED_NUMERIC if not pd.api.types.is_numeric_dtype(df[c])]
assert not bad, f"These columns should be numeric but are not: {bad}. The file may be corrupted - re-download it."
print("Integrity check passed:", df.shape)

Loaded from: diabetic_data.csv
Integrity check passed: (101766, 50)


### 1.3 Display Dataset Records

In [4]:
print("First 5 records:")
display(df.head())

First 5 records:


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [5]:
print("Last 5 records:")
display(df.tail())

Last 5 records:


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
101761,443847548,100162476,AfricanAmerican,Male,[70-80),?,1,3,7,3,MC,?,51,0,16,0,0,0,250.13,291,458,9,None,>8,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,[80-90),?,1,4,5,5,MC,?,33,3,18,0,0,1,560,276,787,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,[70-80),?,1,1,7,1,MC,?,53,0,9,1,0,0,38,590,296,13,None,None,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,[80-90),?,2,3,7,10,MC,Surgery-General,45,2,21,0,0,1,996,285,998,9,None,None,No,No,No,No,No,No,Steady,No,No,Steady,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
101765,443867222,175429310,Caucasian,Male,[70-80),?,1,1,7,6,?,?,13,3,3,0,0,0,530,530,787,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO


In [6]:
print("Random sample of 5 records (random_state=42 so it is repeatable):")
display(df.sample(5, random_state=42))

Random sample of 5 records (random_state=42 so it is repeatable):


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
35956,110939484,19274094,Caucasian,Female,[70-80),?,1,1,6,11,UN,InternalMedicine,68,0,20,0,0,0,250.8,599,263,5,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,NO
60927,170328306,65634327,Caucasian,Male,[50-60),?,1,1,1,1,HM,?,20,0,7,0,0,0,780,427,E941,8,None,None,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
79920,245688426,100657359,Caucasian,Female,[60-70),?,3,6,1,4,HM,?,21,3,23,1,0,2,715,733,724,7,None,None,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
50078,150826224,83144448,Caucasian,Male,[30-40),?,2,1,1,12,CP,Gastroenterology,28,0,19,0,0,1,494,277,117,7,None,None,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,>30
44080,135993852,65234214,AfricanAmerican,Female,[60-70),?,1,2,7,1,?,?,21,0,6,0,0,0,403,584,428,7,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,<30


**Observation (write in your own words after running):** what do the rows represent? What do you notice in the values, for example the `?` symbols?

### 1.4 Display Dataset Information

In [7]:
rows, cols = df.shape
print(f"Number of rows    : {rows:,}")
print(f"Number of columns : {cols}")
print("\nColumn names:")
print(list(df.columns))

Number of rows    : 101,766
Number of columns : 50

Column names:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


In [8]:
# Data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [ ]:
# Missing values as pandas sees them
missing_pandas = df.isna().sum()
print("Missing values detected by pandas (isna):", int(missing_pandas.sum()))

**Check for hidden missing values.** `isna()` only finds real NaN values. Some datasets use a placeholder symbol instead. Let us check whether the symbol `"?"` appears.

In [9]:
# Count "?" placeholders per column (only columns that have them)
question_marks = (df == "?").sum()
question_marks = question_marks[question_marks > 0].sort_values(ascending=False)

hidden_missing = pd.DataFrame({
    "question_mark_count": question_marks,
    "percent_of_rows": (question_marks / len(df) * 100).round(2),
})
display(hidden_missing)

,question_mark_count,percent_of_rows
weight,98569,96.86
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


**Observation (write in your own words):** compare the two results above. Why is it a problem that `isna()` reported a different number than the `?` count?

### 1.5 Check Dataset Shape

In [10]:
print("Shape (rows, columns):", df.shape)
print(f"Number of Rows    : {df.shape[0]:,}")
print(f"Number of Columns : {df.shape[1]}")

Shape (rows, columns): (101766, 50)
Number of Rows    : 101,766
Number of Columns : 50


### 1.6 Identify Column Types
Pandas data types are not always the *real* meaning of a column. Three columns are stored as numbers but are actually **IDs or codes** (`admission_type_id`, `discharge_disposition_id`, `admission_source_id`). Calculating an average of a code has no meaning, so they are treated as categorical. The two ID columns (`encounter_id`, `patient_nbr`) only identify records and are not analysed statistically.

In [11]:
id_cols = ["encounter_id", "patient_nbr"]
coded_categorical = ["admission_type_id", "discharge_disposition_id", "admission_source_id"]

# Numeric columns that are real measurements/counts
numeric_cols = [c for c in df.select_dtypes(include="number").columns
                if c not in id_cols + coded_categorical]

# Text columns are categorical, plus the numeric-looking codes above
categorical_cols = list(df.select_dtypes(exclude="number").columns) + coded_categorical

# Date/time columns (none expected in this dataset)
datetime_cols = list(df.select_dtypes(include="datetime").columns)

print(f"ID columns ({len(id_cols)}):", id_cols)
print(f"\nNumerical columns ({len(numeric_cols)}):", numeric_cols)
print(f"\nCategorical columns ({len(categorical_cols)}):")
print(categorical_cols)
print(f"\nDate/time columns ({len(datetime_cols)}):", datetime_cols if datetime_cols else "None found")

ID columns (2): ['encounter_id', 'patient_nbr']

Numerical columns (8): ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Categorical columns (40):
['race', 'gender', 'age', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id']

Date/time columns (0): None found


**Why column types matter (complete in your own words):** numerical columns support mean, median, standard deviation and correlation; categorical columns support counts, mode and group comparisons. Choosing the wrong type leads to wrong statistics and wrong charts.

*Note:* `age` is stored as text brackets such as `[70-80)`. It is an **ordered** categorical variable. We deal with it in Part 2.

### Part 1 Summary (fill in after running)
- Rows:
- Columns:
- Real missing values found by pandas:
- Hidden missing values (`?`) found in these columns:
- Numerical columns:
- Categorical columns:
- Date/time columns:
- Issues noticed that must be handled in Part 2:

---
## PART 2 – Data Cleaning & Preprocessing

**Approach:** the raw DataFrame `df` is never changed. All cleaning is done on a copy called `df_clean`, so the original can always be compared with the cleaned version.

A running **cleaning log** records every action (what, how many rows/values, why). It becomes the evidence table for the report.

In [12]:
df_clean = df.copy()
rows_before = len(df_clean)

cleaning_log = []   # each entry: step, action, rows_or_values_affected, reason

def log(step, action, affected, reason):
    cleaning_log.append({"step": step, "action": action, "affected": affected, "reason": reason})

print("Working copy created. Rows:", rows_before)

Working copy created. Rows: 101766


### 2.1 Check Missing Values
Step 1: convert the hidden `"?"` placeholders into real missing values (NaN) so pandas can count and handle them.
Step 2: report which columns are affected, how many values, and what percentage.

The text `"None"` in `max_glu_serum` and `A1Cresult` is **not** converted, because it means "test not performed" (real information).

In [13]:
n_question = int((df_clean == "?").sum().sum())
df_clean = df_clean.replace("?", np.nan)

missing = df_clean.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_table = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": (missing / len(df_clean) * 100).round(2),
})
display(missing_table)
print(f"Total '?' converted to NaN: {n_question:,}")
log("2.1", "Converted '?' placeholders to NaN", n_question, "Pandas cannot count or handle '?' as missing")

,missing_count,missing_percent
weight,98569,96.86
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


Total '?' converted to NaN: 192,849


In [14]:
# Proof that "None" is a real category, not missing data
for col in ["max_glu_serum", "A1Cresult"]:
    print(col)
    print(df_clean[col].value_counts(dropna=False).to_string(), "\n")

max_glu_serum
max_glu_serum
None    96420
Norm     2597
>200     1485
>300     1264 

A1Cresult
A1Cresult
None    84748
>8       8216
Norm     4990
>7       3812 



**Observation (write in your own words):** which columns have the most missing values? Do the percentages suggest different treatment for different columns?

### 2.2 Handle Missing Values
Each column gets its own decision, based on how much is missing and what the missing value probably means.

| Column | Missing | Decision | Reason |
|---|---|---|---|
| `weight` | about 97% | **Drop the column** | Almost empty; filling it would invent data. Any statistic would be unreliable. |
| `medical_specialty` | about 49% | Fill with `"Unknown"` | Too many missing to delete rows; the mode would invent a specialty for half the patients. |
| `payer_code` | about 40% | Fill with `"Unknown"` | Same reasoning; the payer is a category and cannot be averaged. |
| `race` | about 2% | Fill with `"Unknown"` | Small amount; guessing a race with the mode would be wrong and unfair to individuals. |
| `diag_2`, `diag_3` | under 2% | Fill with `"Not recorded"` | Checked below: they are mostly missing when the patient had very few diagnoses, so "no additional diagnosis" is the likely meaning. |
| `diag_1` | 21 rows | **Remove those rows** | The primary diagnosis is a core field. These patients had other diagnoses recorded, so it is truly missing. 21 rows is a negligible loss. |

Mean and median cannot be used here because these columns are categories (text), not numbers. Forward/backward fill is not suitable because the rows are not ordered in time.

In [15]:
# Evidence: are diag_2 / diag_3 missing because the patient had few diagnoses?
for col in ["diag_2", "diag_3"]:
    miss_rows = df_clean[df_clean[col].isna()]
    print(f"{col}: {len(miss_rows)} missing. Their number_diagnoses values (smallest first):")
    print(miss_rows["number_diagnoses"].value_counts().sort_index().head(5).to_string(), "\n")

print("diag_1 missing rows - number_diagnoses values:")
print(df_clean[df_clean["diag_1"].isna()]["number_diagnoses"].value_counts().sort_index().to_string())

diag_2: 358 missing. Their number_diagnoses values (smallest first):
number_diagnoses
1    219
2      2
3     11
4     17
5     63 

diag_3: 1423 missing. Their number_diagnoses values (smallest first):
number_diagnoses
1     219
2    1023
3      12
4      24
5     108 

diag_1 missing rows - number_diagnoses values:
number_diagnoses
4    3
5    2
6    2
7    3
8    8
9    3


In [16]:
# 1) Drop the weight column
n = int(df_clean["weight"].isna().sum())
df_clean = df_clean.drop(columns=["weight"])
log("2.2", "Dropped column 'weight'", n, "About 97% missing; cannot be filled reliably")

# 2) Fill categorical columns with an explicit label
fill_unknown = ["medical_specialty", "payer_code", "race"]
for col in fill_unknown:
    n = int(df_clean[col].isna().sum())
    df_clean[col] = df_clean[col].fillna("Unknown")
    log("2.2", f"Filled '{col}' missing with 'Unknown'", n, "Keeps the rows; avoids inventing a value")

for col in ["diag_2", "diag_3"]:
    n = int(df_clean[col].isna().sum())
    df_clean[col] = df_clean[col].fillna("Not recorded")
    log("2.2", f"Filled '{col}' missing with 'Not recorded'", n, "Mostly patients with few diagnoses")

# 3) Remove rows with no primary diagnosis
n = int(df_clean["diag_1"].isna().sum())
df_clean = df_clean.dropna(subset=["diag_1"])
log("2.2", "Removed rows with missing 'diag_1'", n, "Primary diagnosis is essential; only 21 rows")

print("Remaining missing values:", int(df_clean.isna().sum().sum()))
print("Shape now:", df_clean.shape)

Remaining missing values: 0
Shape now: (101745, 49)


### 2.3 Check Duplicate Records
Two checks: completely identical rows, and repeated `encounter_id` (each hospital visit should appear once).
We also look at **repeated patients**: the same person can have many visits. These are *not* duplicates, because each row is a different hospital stay, so they are kept. It is a limitation to mention: visits from the same patient are not fully independent.

In [17]:
dup_rows = int(df_clean.duplicated().sum())
dup_encounters = int(df_clean["encounter_id"].duplicated().sum())
print("Fully identical duplicate rows :", dup_rows)
print("Repeated encounter_id          :", dup_encounters)

unique_patients = df_clean["patient_nbr"].nunique()
print(f"\nUnique patients: {unique_patients:,} for {len(df_clean):,} visits")
print("Most visits by one patient:", df_clean["patient_nbr"].value_counts().max())

# Remove duplicates only if they exist
if dup_rows > 0:
    df_clean = df_clean.drop_duplicates()
    log("2.3", "Removed duplicate rows", dup_rows, "Identical records")
else:
    log("2.3", "No duplicate rows found - nothing removed", 0, "Checked with duplicated()")

# Verify
print("\nDuplicates after check:", int(df_clean.duplicated().sum()))

Fully identical duplicate rows : 0
Repeated encounter_id          : 0

Unique patients: 71,509 for 101,745 visits
Most visits by one patient: 40

Duplicates after check: 0


**Observation (write in your own words):** why are repeated patients not treated as duplicates here? How could they affect later conclusions?

### 2.4 Check Data Types
Problems to look for: numbers stored as text, dates stored as strings, categories stored as numbers.
- The 3 `*_id` code columns are numbers but are really categories -> convert to text categories.
- Text columns -> `category` dtype (less memory, and correct meaning).
- `age` -> **ordered** category (youngest to oldest) so charts sort correctly.
- No date columns exist, so there is nothing to convert to datetime.

In [18]:
print("Data types before conversion:")
print(df_clean.dtypes.astype(str).value_counts().to_string())

# 1) Numeric codes -> categories
for col in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
    df_clean[col] = df_clean[col].astype(str).astype("category")

# 2) Age: ordered category
age_order = ["[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
             "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"]
assert set(df_clean["age"].unique()) <= set(age_order), "Unexpected age bracket found"
df_clean["age"] = pd.Categorical(df_clean["age"], categories=age_order, ordered=True)

# 3) Remaining text columns -> category
text_cols = [c for c in df_clean.select_dtypes(include=["object", "string"]).columns
             if c not in ["encounter_id", "patient_nbr"]]
for col in text_cols:
    df_clean[col] = df_clean[col].astype("category")

log("2.4", "Converted 3 code columns, age (ordered) and text columns to category", len(text_cols) + 4,
    "Codes are not quantities; age brackets have a natural order")

print("\nData types after conversion:")
print(df_clean.dtypes.astype(str).value_counts().to_string())

Data types before conversion:
object    36
int64     13

Data types after conversion:
category    39
int64       10


### 2.5 Rename Columns
Some names are unclear or awkward to use in code:
- `patient_nbr` -> `patient_id`
- `diag_1`, `diag_2`, `diag_3` -> `primary_diagnosis`, `secondary_diagnosis`, `additional_diagnosis`
- `max_glu_serum` -> `max_glucose_serum`, `A1Cresult` -> `a1c_result`, `diabetesMed` -> `diabetes_med` (consistent lowercase_with_underscores)
- Medicine columns with a hyphen (e.g. `glyburide-metformin`) -> underscore, because a hyphen is read as a minus sign in Python code.

Values are also made clearer: the label `"None"` in `max_glucose_serum` and `a1c_result` becomes `"Not tested"`, because "None" looks like a missing value but means the test was not done.

In [19]:
rename_map = {
    "patient_nbr": "patient_id",
    "diag_1": "primary_diagnosis",
    "diag_2": "secondary_diagnosis",
    "diag_3": "additional_diagnosis",
    "max_glu_serum": "max_glucose_serum",
    "A1Cresult": "a1c_result",
    "diabetesMed": "diabetes_med",
    "glyburide-metformin": "glyburide_metformin",
    "glipizide-metformin": "glipizide_metformin",
    "glimepiride-pioglitazone": "glimepiride_pioglitazone",
    "metformin-rosiglitazone": "metformin_rosiglitazone",
    "metformin-pioglitazone": "metformin_pioglitazone",
}
# Safety check: every old name must really exist
missing_names = [c for c in rename_map if c not in df_clean.columns]
assert not missing_names, f"These columns do not exist: {missing_names}"

df_clean = df_clean.rename(columns=rename_map)

# Make the "None" label clearer
for col in ["max_glucose_serum", "a1c_result"]:
    df_clean[col] = df_clean[col].cat.rename_categories({"None": "Not tested"})

log("2.5", "Renamed 12 columns; relabelled 'None' as 'Not tested' in 2 columns", 12,
    "Clearer, consistent names; avoids confusing 'None' with missing")
print(list(df_clean.columns))

['encounter_id', 'patient_id', 'race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'primary_diagnosis', 'secondary_diagnosis', 'additional_diagnosis', 'number_diagnoses', 'max_glucose_serum', 'a1c_result', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide_metformin', 'glipizide_metformin', 'glimepiride_pioglitazone', 'metformin_rosiglitazone', 'metformin_pioglitazone', 'change', 'diabetes_med', 'readmitted']


### 2.6 Check Invalid Values
Look for values that make no logical sense. Unusual is **not** the same as invalid: a very long stay or many medications can be real. We only remove values that are impossible, and we inspect the unusual ones first.

In [20]:
# 1) Impossible values in count columns (negative numbers, zero-day stays)
count_cols = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
              "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
print("Negative values per column:")
print((df_clean[count_cols] < 0).sum().to_string())
print("\nHospital stays shorter than 1 day:", int((df_clean["time_in_hospital"] < 1).sum()))

# 2) Valid categories
print("\nGender values:")
print(df_clean["gender"].value_counts().to_string())

# 3) Range of each numeric column (min and max)
display(df_clean[count_cols].agg(["min", "max"]).T)

Negative values per column:
time_in_hospital      0
num_lab_procedures    0
num_procedures        0
num_medications       0
number_outpatient     0
number_emergency      0
number_inpatient      0
number_diagnoses      0

Hospital stays shorter than 1 day: 0

Gender values:
gender
Female             54703
Male               47039
Unknown/Invalid        3


,min,max
time_in_hospital,1,14
num_lab_procedures,1,132
num_procedures,0,6
num_medications,1,81
number_outpatient,0,42
number_emergency,0,76
number_inpatient,0,21
number_diagnoses,1,16


**Decision on gender:** the value `"Unknown/Invalid"` is not a usable category, and gender cannot be guessed. There are only a few such rows, so they are removed (explained in the log).

In [21]:
n = int((df_clean["gender"] == "Unknown/Invalid").sum())
df_clean = df_clean[df_clean["gender"] != "Unknown/Invalid"].copy()
df_clean["gender"] = df_clean["gender"].cat.remove_unused_categories()
log("2.6", "Removed rows with gender 'Unknown/Invalid'", n, "Not a valid category; cannot be inferred; negligible row count")
print("Rows removed:", n, "| Shape now:", df_clean.shape)

Rows removed: 3 | Shape now: (101742, 49)


**Outliers are inspected, not deleted.** Below we count how many values fall outside the usual range using the IQR rule (a common statistical rule). We keep them: they are plausible hospital data, and removing them would bias the results. We will examine them again in Part 3 (EDA).

In [22]:
def iqr_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((series < low) | (series > high)).sum()), low, high

rows = []
for col in count_cols:
    n_out, low, high = iqr_outliers(df_clean[col])
    rows.append({"column": col, "values_outside_IQR_range": n_out,
                 "percent": round(n_out / len(df_clean) * 100, 2),
                 "lower_limit": round(low, 1), "upper_limit": round(high, 1),
                 "max_value": df_clean[col].max()})
display(pd.DataFrame(rows).set_index("column"))
log("2.6", "Outliers inspected but kept", "see table", "Plausible real values; removal would bias results")

,values_outside_IQR_range,percent,lower_limit,upper_limit,max_value
column,,,,,
time_in_hospital,2252,2.21,-4.0,12.0,14
num_lab_procedures,143,0.14,-8.0,96.0,132
num_procedures,4954,4.87,-3.0,5.0,6
num_medications,2557,2.51,-5.0,35.0,81
number_outpatient,16737,16.45,0.0,0.0,42
number_emergency,11379,11.18,0.0,0.0,76
number_inpatient,7046,6.93,-1.5,2.5,21
number_diagnoses,281,0.28,1.5,13.5,16


**Note on reading this table:** the IQR rule works poorly for columns where most values are 0 (the visit-count columns). When the upper limit is 0 or very small, almost any non-zero value is flagged, even though a patient with one prior visit is completely normal. A flagged value is a *reason to look*, not proof of an error.

**SHOULD DO – check before Part 3 (read `description.pdf`):** some `discharge_disposition_id` codes may represent patients who died or went to hospice. Such patients cannot be readmitted, so including them can distort any readmission analysis. I have not confirmed which code numbers these are. Find the ID mapping in the dataset documentation, check it, and decide whether to exclude them. Write your decision here:

*(your note)*

### 2.7 Verify the Cleaned Dataset

In [23]:
print("=== Cleaned dataset verification ===")
print("Missing values  :", int(df_clean.isna().sum().sum()))
print("Duplicate rows  :", int(df_clean.duplicated().sum()))
print(f"Shape before    : {df.shape}")
print(f"Shape after     : {df_clean.shape}")
print(f"Rows removed    : {rows_before - len(df_clean)}  ({(rows_before - len(df_clean)) / rows_before * 100:.3f}%)")
print(f"Columns removed : {df.shape[1] - df_clean.shape[1]}")
print("\nData types:")
print(df_clean.dtypes.astype(str).value_counts().to_string())

=== Cleaned dataset verification ===
Missing values  : 0
Duplicate rows  : 0
Shape before    : (101766, 50)
Shape after     : (101742, 49)
Rows removed    : 24  (0.024%)
Columns removed : 1

Data types:
category    39
int64       10


In [24]:
# Basic statistics of the numeric columns
display(df_clean[count_cols].describe().T.round(2))

,count,mean,std,min,25%,50%,75%,max
time_in_hospital,101742.0,4.40,2.99,1.0,2.0,4.0,6.0,14.0
num_lab_procedures,101742.0,43.10,19.67,1.0,31.0,44.0,57.0,132.0
num_procedures,101742.0,1.34,1.71,0.0,0.0,1.0,2.0,6.0
num_medications,101742.0,16.02,8.13,1.0,10.0,15.0,20.0,81.0
number_outpatient,101742.0,0.37,1.27,0.0,0.0,0.0,0.0,42.0
number_emergency,101742.0,0.20,0.93,0.0,0.0,0.0,0.0,76.0
number_inpatient,101742.0,0.64,1.26,0.0,0.0,0.0,1.0,21.0
number_diagnoses,101742.0,7.42,1.93,1.0,6.0,8.0,9.0,16.0


In [25]:
# Cleaning log - evidence of every action taken
log_table = pd.DataFrame(cleaning_log)
display(log_table)

,step,action,affected,reason
0,2.1,Converted '?' placeholders to NaN,192849,Pandas cannot count or handle '?' as missing
1,2.2,Dropped column 'weight',98569,About 97% missing; cannot be filled reliably
2,2.2,Filled 'medical_specialty' missing with 'Unknown',49949,Keeps the rows; avoids inventing a value
3,2.2,Filled 'payer_code' missing with 'Unknown',40256,Keeps the rows; avoids inventing a value
4,2.2,Filled 'race' missing with 'Unknown',2273,Keeps the rows; avoids inventing a value
5,2.2,Filled 'diag_2' missing with 'Not recorded',358,Mostly patients with few diagnoses
6,2.2,Filled 'diag_3' missing with 'Not recorded',1423,Mostly patients with few diagnoses
7,2.2,Removed rows with missing 'diag_1',21,Primary diagnosis is essential; only 21 rows
8,2.3,No duplicate rows found - nothing removed,0,Checked with duplicated()
9,2.4,"Converted 3 code columns, age (ordered) and te...",39,Codes are not quantities; age brackets have a ...


In [26]:
# Save the cleaned dataset (used later by the Streamlit app)
CLEAN_PATH = "diabetic_data_cleaned.csv"
df_clean.to_csv(CLEAN_PATH, index=False)
print("Saved:", CLEAN_PATH)
# Note: CSV does not store 'category' types, so the app must set them again when loading.

Saved: diabetic_data_cleaned.csv


### Part 2 Summary (fill in after running, in your own words)
- Problems found in the raw data:
- Cleaning decisions and why:
- Rows removed in total and why:
- Columns removed and why:
- What was deliberately **kept** (e.g. outliers, repeated patients) and why: